# 09 — Validações: NorthwindDW DuckDB

14 verificações de integridade — todas via `conn.execute()`.

| # | Check | Esperado |
|---|-------|----------|
| 1-5 | Contagens e grains | 2155, 2155, True, 830, 0 |
| 6-7 | SCD2: 1 IsCurrent por chave natural | 0 violações |
| 8-11 | Integridade referencial FactSales → Dims | 0 órfãos |
| 12 | DimDate cobre OrderDateKeys | True |
| 13 | FactProductStock grain único | 0 duplicatas |
| 14 | NetRevenue ≤ GrossRevenue | 0 violações |

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH

conn = get_conn()
results = []

def check(num, desc, result, expected, check_fn=None):
    ok = check_fn(result, expected) if check_fn else (result == expected)
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] #{num}: {desc}\n       Resultado: {result} | Esperado: {expected}")
    results.append((num, desc, status, result, expected))
    return ok

def sql_count(q):
    return conn.execute(q).fetchone()[0]

print(f"Conectado: {DB_PATH}")

In [ ]:
# Checks 1-5: contagens e grains
n_od = sql_count("SELECT COUNT(*) FROM bronze.order_details")
check(1, "bronze.order_details count", n_od, 2155)

n_fs = sql_count("SELECT COUNT(*) FROM gold.FactSales")
check(2, "gold.FactSales count", n_fs, 2155)
check(3, "FactSales == bronze.order_details", n_fs == n_od, True)

n_fof = sql_count("SELECT COUNT(*) FROM gold.FactOrderFulfillment")
check(4, "gold.FactOrderFulfillment count", n_fof, 830)

dups_fof = sql_count("""
    SELECT COUNT(*) FROM (
        SELECT OrderID FROM gold.FactOrderFulfillment GROUP BY OrderID HAVING COUNT(*) > 1
    )
""")
check(5, "FactOrderFulfillment grain único", dups_fof, 0)

In [ ]:
# Checks 6-7: SCD2 — um único IsCurrent por chave natural
viol_cust = sql_count("""
    SELECT COUNT(*) FROM (
        SELECT CustomerID FROM gold.DimCustomer WHERE IsCurrent = TRUE
        GROUP BY CustomerID HAVING COUNT(*) > 1
    )
""")
check(6, "DimCustomer: 1 IsCurrent por CustomerID", viol_cust, 0)

viol_prod = sql_count("""
    SELECT COUNT(*) FROM (
        SELECT ProductID FROM gold.DimProduct WHERE IsCurrent = TRUE
        GROUP BY ProductID HAVING COUNT(*) > 1
    )
""")
check(7, "DimProduct: 1 IsCurrent por ProductID", viol_prod, 0)

In [ ]:
# Checks 8-11: integridade referencial FactSales → Dims
for num, sk, dim in [
    (8,  "CustomerSK", "gold.DimCustomer"),
    (9,  "ProductSK",  "gold.DimProduct"),
    (10, "EmployeeSK", "gold.DimEmployee"),
    (11, "ShipperSK",  "gold.DimShipper"),
]:
    n = sql_count(f"SELECT COUNT(*) FROM gold.FactSales fs "
                  f"WHERE NOT EXISTS (SELECT 1 FROM {dim} d WHERE d.{sk} = fs.{sk})")
    check(num, f"FactSales: sem órfãos por {sk}", n, 0)

In [ ]:
# Check 12: DimDate cobre todo o intervalo de OrderDateKeys
min_dk   = sql_count("SELECT MIN(OrderDateKey) FROM gold.FactSales")
max_dk   = sql_count("SELECT MAX(OrderDateKey) FROM gold.FactSales")
min_date = sql_count("SELECT MIN(DateKey) FROM gold.DimDate")
max_date = sql_count("SELECT MAX(DateKey) FROM gold.DimDate")
date_ok  = False if None in (min_dk, max_dk, min_date, max_date) else (min_date <= min_dk and max_date >= max_dk)
check(12, f"DimDate cobre OrderDateKeys ({min_dk}..{max_dk})", date_ok, True)

# Check 13: FactProductStock grain único (SnapshotDateKey + ProductSK)
dups_stock = sql_count("""
    SELECT COUNT(*) FROM (
        SELECT SnapshotDateKey, ProductSK FROM gold.FactProductStock
        GROUP BY SnapshotDateKey, ProductSK HAVING COUNT(*) > 1
    )
""")
check(13, "FactProductStock grain único", dups_stock, 0)

# Check 14: NetRevenue nunca maior que GrossRevenue
viol_rev = sql_count("SELECT COUNT(*) FROM gold.FactSales WHERE NetRevenue > GrossRevenue + 0.01")
check(14, "NetRevenue <= GrossRevenue", viol_rev, 0)

In [ ]:
passed = sum(1 for r in results if r[2] == "PASS")
failed = sum(1 for r in results if r[2] == "FAIL")
print("=" * 50)
print(f"Validações: {passed}/{len(results)} PASS | {failed} FAIL")
print("=" * 50)
if failed == 0:
    print("\nTodos os checks passaram! Pipeline DuckDB validado.")
else:
    for r in results:
        if r[2] == "FAIL":
            print(f"  FAIL #{r[0]}: {r[1]} → {r[3]} (esperado: {r[4]})")

conn.close()